In [13]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

In [14]:
def load_session(filepath):
    df = pd.read_csv(filepath)
    
    # drop scroll events
    df = df[df['button'] != 'Scroll']
    
    # drop 65535 coordinate artifacts
    df = df[~((df['x'] == 65535) | (df['y'] == 65535))]
    
    # sort by record timestamp
    df = df.sort_values('record timestamp').reset_index(drop=True)
    
    return df

In [21]:
cleaned_sessions = load_session('DataSamples/session_2144641057.txt')
print(f"Total rows: {cleaned_sessions.shape[0]}")
print(cleaned_sessions.head())
print("\n")

# event distribution
print(cleaned_sessions['state'].value_counts())
print(cleaned_sessions['button'].value_counts())
print("\n")

# left clicks
left_clicks = cleaned_sessions[
    (cleaned_sessions['button'] == 'Left') & 
    (cleaned_sessions['state'] == 'Pressed')
]
print(f"Total left clicks: {len(left_clicks)}")
print("\n")

# timestamp gaps
moves = cleaned_sessions[cleaned_sessions['state'] == 'Move'].copy()
moves['dt'] = moves['record timestamp'].diff()
print(f"Median gap: {moves['dt'].median()*1000:.1f}ms")
print(f"Gaps over 2s: {len(moves[moves['dt'] > 2.0])}")
print(f"Largest gap: {moves['dt'].max():.1f}s")
print("\n")

# coordinate range
print(f"X range: {cleaned_sessions['x'].min()} to {cleaned_sessions['x'].max()}")
print(f"Y range: {cleaned_sessions['y'].min()} to {cleaned_sessions['y'].max()}")


Total rows: 29675
   record timestamp  client timestamp    button state     x    y
0             0.000             0.000  NoButton  Move  1043  410
1             0.232             0.094  NoButton  Move  1024  410
2             0.232             0.094  NoButton  Move  1019  410
3             0.232             0.234  NoButton  Move   979  409
4             0.342             0.234  NoButton  Move   974  411


state
Move        26257
Drag         1258
Pressed      1080
Released     1080
Name: count, dtype: int64
button
NoButton    27515
Left         2148
Right          12
Name: count, dtype: int64


Total left clicks: 1074


Median gap: 113.0ms
Gaps over 2s: 678
Largest gap: 686.4s


X range: 0 to 1919
Y range: 0 to 1079


In [ ]:
def extract_trajectory(df):
    trajectory = []
    for _, row in df.iterrows():
        event = {
            'timestamp': row['record timestamp'],
            'x': row['x'],
            'y': row['y'],
            'button': row['button'],
            'state': row['state']
        }
        trajectory.append(event)